# SKKU Multimodal AI Bias Challenge

Builds the self-constructed training set for DACON 236722. Input = image + context + question,
output = 3-choice label (0/1/2). Core goal: **avoid social bias + pick "Unknown" when context is insufficient**.

**Strategy (hybrid)**: SB-Bench real MCQ (primary, real images) + BBQ text (9-axis augmentation).

This notebook **runs the existing project source** (`src/*.py`) — it does not re-define the code.
Get the project onto Colab (Google Drive mount or zip upload), then run the stages.

> ⚠️ SB-Bench is **CC BY-NC 4.0 (non-commercial)** and **HF-gated**. Accept the terms and use a Read token.
Per-sample source/license is recorded in `data/metadata.jsonl`.

## 1. Get the project source

**Option A — Google Drive (recommended):** upload the project folder to your Drive, then set `PROJECT_DIR`.
**Option B — Zip upload:** zip the project locally, upload it here, and it will be extracted.
Run only one of the two cells below.

In [ ]:
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
# 👇 edit to your project path on Drive
PROJECT_DIR = '/content/drive/MyDrive/Multi-Modal-AI-Bias-Challenge'
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())
print('src files:', sorted(os.listdir('src')))

In [ ]:
# Option B: zip upload (run instead of Option A)
# from google.colab import files
# import zipfile, os
# up = files.upload()                      # choose project.zip
# name = next(iter(up))
# with zipfile.ZipFile(name) as z: z.extractall('/content/project')
# # if the zip has a top-level folder, chdir into it
# root = '/content/project'
# inner = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
# PROJECT_DIR = os.path.join(root, inner[0]) if len(inner) == 1 else root
# os.chdir(PROJECT_DIR); print('cwd:', os.getcwd())

## 2. Install dependencies (no GPU needed)

In [ ]:
!pip -q install -r requirements.txt
print('deps installed')

## 3. HuggingFace token (SB-Bench gate)

1. Accept terms at https://huggingface.co/datasets/ucf-crcv/SB-Bench
2. Create a **Read** token at https://huggingface.co/settings/tokens
3. Run the cell and paste it (or store it in Colab Secrets as `HF_TOKEN`).

In [ ]:
import os, getpass
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('HF_TOKEN (read): ').strip()
os.environ['HF_TOKEN'] = token
open('.env', 'w').write(f'HF_TOKEN={token}\n')   # src.common.load_config reads this
from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print('HF login OK')

## 4. (Optional) test.csv for leak removal + proportional Unknown distribution

If `data/raw/test/test.csv` (from the competition `open.zip`) is present, the pipeline removes test
leakage and reproduces the observed Unknown-term distribution. Otherwise it falls back to uniform Unknown
and skips leak removal.

In [ ]:
import os
if os.path.exists('data/raw/test/test.csv'):
    print('found data/raw/test/test.csv')
else:
    print('test.csv not found — upload it (optional):')
    try:
        from google.colab import files
        os.makedirs('data/raw/test', exist_ok=True)
        up = files.upload()
        for name, content in up.items():
            open('data/raw/test/test.csv', 'wb').write(content)
            print('saved data/raw/test/test.csv')
    except Exception as e:
        print('skipped upload:', e)

## 5. Run the pipeline
Order: `map_sbbench → augment_bbq → compose → metadata → validate`.

### 5.1 SB-Bench mapping + image save (~12.3GB download; minutes to tens of minutes)

In [ ]:
!python -m src.map_sbbench

### 5.2 Unit tests (pure functions)

In [ ]:
!python -m pytest tests/ -q

### 5.3 BBQ augmentation (under-filled cells)

In [ ]:
!python -m src.augment_bbq

### 5.4 Compose (Unknown re-diversify / position balancing → train.csv)

In [ ]:
!python -m src.compose

### 5.5 Test-leak removal + metadata

In [ ]:
!python -m src.metadata

### 5.6 Validate (schema / distribution / bias score / reproducibility / baseline smoke)

In [ ]:
!python -m src.validate

### 5.7 Validate OOD split (leave-axis-out 무결성/분포)

`config.yaml`의 `ood_axes`(기본 `Religion`, `Sexual_orientation`)를 통째로 hold-out한 OOD 검증셋을 재현해, train / in-domain-val / ood-val 3분할의 무결성(sample_id 겹침=0, OOD가 지정 축만 포함)과 축·극성 분포를 리포트한다. 학습이 OOD 기준으로 best 체크포인트를 고르는 전제를 보장한다.

In [ ]:
!python -m src.validate --ood

## 6. Inspect + download results

In [ ]:
import pandas as pd, json, yaml
from collections import Counter
df = pd.read_csv('data/processed/train/train.csv')
print('train rows:', len(df), '| columns:', list(df.columns))
print(df.head(3).to_string())
lex = set(yaml.safe_load(open('config.yaml'))['unknown_lexicon'])
pos = Counter()
for a in df['answers']:
    for i, o in enumerate(json.loads(a)):
        if o in lex:
            pos[i] += 1
print('Unknown position counts (0/1/2):', dict(sorted(pos.items())))

In [ ]:
# Zip train.csv + images + metadata, then download
!cd data/processed && zip -qr /content/train_dataset.zip train
!cp data/metadata.jsonl /content/ 2>/dev/null || true
try:
    from google.colab import files
    files.download('/content/train_dataset.zip')
except Exception as e:
    print('download manually at /content/train_dataset.zip', e)